In [1]:
import requests
import json
import os
import fitz  # PyMuPDF
from pathlib import Path

def extract_text_from_bytes(file_bytes, filename, api_key, model, task_type, max_tokens, temperature, top_p, repetition_penalty, prompt_text):
    url = "https://api.opentyphoon.ai/v1/ocr"

    files = {'file': (filename, file_bytes, 'image/jpeg')}
    data = {
        'model': model,
        'task_type': task_type,
        'max_tokens': str(max_tokens),
        'temperature': str(temperature),
        'top_p': str(top_p),
        'repetition_penalty': str(repetition_penalty),
        'prompt': prompt_text
    }

    headers = {'Authorization': f'Bearer {api_key}'}

    try:
        response = requests.post(url, files=files, data=data, headers=headers)
        if response.status_code == 200:
            result = response.json()
            extracted_texts = []
            for page_result in result.get('results', []):
                if page_result.get('success') and page_result.get('message'):
                    content = page_result['message']['choices'][0]['message']['content']
                    try:
                        parsed_content = json.loads(content)
                        text = parsed_content.get('natural_text', content)
                    except json.JSONDecodeError:
                        text = content
                    extracted_texts.append(text)
                elif not page_result.get('success'):
                    print(f"Error processing {page_result.get('filename', 'unknown')}: {page_result.get('error', 'Unknown error')}")
            return '\n'.join(extracted_texts)
        else:
            print(f"Error: {response.status_code}")
            print(response.text)
            return None
    except Exception as e:
        print(f"Exception during OCR request for {filename}: {e}")
        return None

def get_page_mapping(txt_path, is_bch):
    """
    อ่านไฟล์ txt และสร้าง Mapping: หน้าที่เท่าไหร่ ตรงกับ หน่วยที่เท่าไหร่
    """
    with open(txt_path, 'r', encoding='utf-8') as f:
        numbers = [int(line.strip()) for line in f if line.strip().isdigit()]

    if not numbers:
        return {}, []

    expected = []
    if is_bch:
        pattern = [1, 1, 2]
        idx = 0
        current = numbers[0]
        max_num = numbers[-1]
        while current <= max_num:
            expected.append(current)
            current += pattern[idx % 3]
            idx += 1
    else:
        # ถ้ามีหน้าเดียวให้ step=1
        step = numbers[1] - numbers[0] if len(numbers) > 1 else 1
        current = numbers[0]
        max_num = numbers[-1]
        while current <= max_num:
            expected.append(current)
            current += step

    page_to_unit = {}
    for expected_idx, page_num in enumerate(expected):
        if is_bch:
            unit = (expected_idx // 3) + 1  # 3 หน้า ต่อ 1 หน่วย
        else:
            unit = expected_idx + 1         # 1 หน้า ต่อ 1 หน่วย
        page_to_unit[page_num] = unit

    return page_to_unit, numbers

def process_pdf(pdf_path, api_key, model, task_type, max_tokens, temperature, top_p, repetition_penalty, prompt_text, pages_to_process, page_to_unit):
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"Could not open {pdf_path}: {e}")
        return {}

    # เก็บผลลัพธ์แยกตามหน่วย
    unit_results = {}

    for page_num in pages_to_process:
        # page_num เริ่มที่ 1 แต่ load_page ของ PyMuPDF เริ่มที่ 0
        page_idx = page_num - 1 
        
        if page_idx >= len(doc):
            print(f"    Warning: Page {page_num} is out of range.")
            continue

        unit = page_to_unit.get(page_num, "Unknown")
        print(f"  - Extracting page {page_num} (Unit {unit})")
        
        page = doc.load_page(page_idx)
        zoom = 2.0
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat)

        img_bytes = pix.tobytes("jpg")
        
        while len(img_bytes) > 4.5 * 1024 * 1024 and zoom > 0.5:
            zoom -= 0.3
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat)
            img_bytes = pix.tobytes("jpg")
        print("start extract")
        page_text = extract_text_from_bytes(
            img_bytes, f"{pdf_path.stem}_p{page_num}.jpg", api_key, model,
            task_type, max_tokens, temperature, top_p, repetition_penalty, prompt_text
        )

        if page_text:
            if unit not in unit_results:
                unit_results[unit] = []
            unit_results[unit].append(f"--- Page {page_num} ---\n{page_text}")
        else:
            print(f"    Failed to extract text from page {page_num}")

    doc.close()
    return unit_results

def main():
    api_key = os.environ.get('TOKEN_KEY')
    if not api_key:
        if os.path.exists('.env'):
            with open('.env', 'r', encoding='utf-8') as f:
                for line in f:
                    if line.startswith('TOKEN_KEY='):
                        api_key = line.split('=')[1].strip()
                        break

    if not api_key:
        print("Error: TOKEN_KEY not found in environment or .env file.")
        return

    model = "typhoon-ocr"
    task_type = "default"
    max_tokens = 16384
    temperature = 0.05
    top_p = 0.3
    repetition_penalty = 1
    
    i_prompt = """Role: Specialized Data Extraction Expert
You are an expert in parsing Thai Election Report OCR text and converting it into highly structured data formats. Your objective is to process raw OCR text and output exactly two specific CSV tables.

1. Data Identification & Analysis
Type "บช" (Party List): Identified by keywords like "แบบบัญชีรายชื่อ" or "(บช)".
Type "เขต" (Constituency): Identified by keywords like "แบบแบ่งเขตเลือกตั้ง".

Location Data: Extract names following these keywords:
จังหวัด (Province)
อำเภอ (District)
ตำบล (Sub-district)
หน่วยที่ (Unit Number)

2. Strict Extraction & Formatting Rules
Value Integrity:
- Convert Thai numerals (๐-๙) to Arabic numerals (0-9) before extraction. (thai_digits = "๐๑๒๓๔๕๖๗๘๙")
- If a numeric field is missing, unreadable, or invalid → use -1
- If explicitly zero → use 0
- Prioritize Arabic numerals

Location Cleanup:
- Remove prefixes (จังหวัด, อำเภอ, ตำบล, หน่วยที่)

Language: Keep Thai names

3. Output Formats

CSV 1:
type,province,district,sub-district,unit_number,name,score

Consistency Check:
- Ensure:
  บัตรดี = sum(score) ของพรรคหรือสสเขต
- If not:
  - Re-check extraction
  - If still inconsistent → assign -1 to unreliable fields

4. CRITICAL:
- Output ONLY one Markdown CSV blocks
- First = CSV1
- No explanation
"""

    source_dir = Path('ลำปาง')
    txt_source_dir = Path('page-ocr') # โฟลเดอร์ที่เก็บไฟล์ txt
    output_dir = Path('compress-ลำปาง')
    output_dir.mkdir(exist_ok=True)

    pdf_files = sorted(list(source_dir.glob('**/*.pdf')))
    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_path in pdf_files:
        # ค้นหาไฟล์ txt ที่มีชื่อตรงกับไฟล์ PDF
        txt_matches = list(txt_source_dir.rglob(f"{pdf_path.stem}.txt"))
        
        if not txt_matches:
            print(f"⏭️ Skipping {pdf_path.name}: ไม่พบไฟล์ .txt ในโฟลเดอร์ page-ocr")
            continue
            
        txt_path = txt_matches[0]
        is_bch = "บช" in pdf_path.stem
        
        # ดึง Mapping ว่าหน้าไหนเป็นของหน่วยที่เท่าไหร่
        page_to_unit, pages_to_process = get_page_mapping(txt_path, is_bch)
        
        if not pages_to_process:
            print(f"⏭️ Skipping {pdf_path.name}: ไฟล์ .txt ว่างเปล่า")
            continue

        print(f"\nProcessing: {pdf_path}")
        
        # ฟังก์ชัน process_pdf จะคืนค่ากลับมาเป็น Dictionary โดยมี Key เป็นเลขหน่วย
        unit_extracted_dict = process_pdf(
            pdf_path, api_key, model, task_type, max_tokens,
            temperature, top_p, repetition_penalty, i_prompt,
            pages_to_process, page_to_unit
        )

        # บันทึกแยกไฟล์ตามหน่วยที่ (Unit Number)
        for unit, texts in unit_extracted_dict.items():
            output_filename = f"{pdf_path.stem}_Unit_{unit}.txt"
            output_path = output_dir / output_filename
            
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write("\n\n".join(texts))
                
            print(f"  ✅ Successfully saved Unit {unit} to {output_filename}")

In [2]:
main()

Found 94 PDF files to process.

Processing: ลำปาง\นอกเขต\ชุดที่ 1\ชุดที่ 1 ส.ส. 5-17(บช).pdf
  - Extracting page 1 (Unit 1)
start extract
Error processing ชุดที่ 1 ส.ส. 5-17(บช)_p1.jpg: Error processing file: litellm.Timeout: Timeout Error: OpenAIException - <html>
<head><title>504 Gateway Time-out</title></head>
<body>
<center><h1>504 Gateway Time-out</h1></center>
</body>
</html>
    Failed to extract text from page 1
  - Extracting page 2 (Unit 1)
start extract
  - Extracting page 3 (Unit 1)
start extract
  ✅ Successfully saved Unit 1 to ชุดที่ 1 ส.ส. 5-17(บช)_Unit_1.txt

Processing: ลำปาง\นอกเขต\ชุดที่ 1\ชุดที่ 1 ส.ส. 5-17.pdf
  - Extracting page 1 (Unit 1)
start extract
  ✅ Successfully saved Unit 1 to ชุดที่ 1 ส.ส. 5-17_Unit_1.txt

Processing: ลำปาง\นอกเขต\ชุดที่ 10\ชุดที่ 10 5-17(บช).pdf
  - Extracting page 1 (Unit 1)
start extract
  - Extracting page 2 (Unit 1)
start extract
  - Extracting page 3 (Unit 1)
start extract
  ✅ Successfully saved Unit 1 to ชุดที่ 10 5-17(บช)_Unit_1.

KeyboardInterrupt: 